In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import hydra

from genpp import BASE_DIR
from genpp.configs import register_resolvers

try:
    register_resolvers()
except ValueError:
    pass

# Test if Model works in general

In [ ]:
with hydra.initialize_config_dir(version_base=None, config_dir=str(BASE_DIR / "configs")):
    cfg = hydra.compose(config_name="base_chen_spatial")

In [ ]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup("fit")
dataloader = datamodule.train_dataloader()

In [ ]:
cfg.model.n_samples = (
    16  # for testing purposes only (this only influences the predict step for the fm models)
)

In [ ]:
model = hydra.utils.instantiate(
    cfg.model,
    rescaler=datamodule.y_reverseModules if cfg.model.use_rescaler else None,
    internal_td_scaling="learned",
)

In [ ]:
trainer = hydra.utils.instantiate(
    cfg.trainer,
    logger=None,
    detect_anomaly=True,
    accelerator="cpu",
    max_epochs=10,
    overfit_batches=10,
    check_val_every_n_epoch=5,
)
trainer.fit(model, datamodule=datamodule)

## Try to load the model from checkpoint

In [ ]:
import os
from pathlib import Path

from genpp.models.cgm.chen import CNNChenModel

cwd = Path(os.getcwd())
checkpoint_path = list((cwd / "lightning_logs" / "version_0" / "checkpoints").glob("*.ckpt"))[0]
checkpoint_path

In [ ]:
for batch in dataloader:
    break

In [ ]:
mod_new = CNNChenModel.load_from_checkpoint(checkpoint_path)

In [ ]:
res = mod_new.predict_step(batch)

## Some random short tests

In [ ]:
import torch
from einops import rearrange

from genpp.models.loss import PatchwiseEnergyScore

t = torch.randn(64, 2, 30, 30)
unfold = torch.nn.Unfold(kernel_size=(3, 3), stride=1)
patches = unfold(t)

In [ ]:
pes = PatchwiseEnergyScore(height=30, width=30, patch_size=3)

In [ ]:
t1 = torch.randint(0, 10, (64, 10, 1, 30, 30))
t2 = torch.randint(0, 10, (64, 10, 1, 30, 30)) / 10.0
t_c = torch.cat([t1, t2], dim=2)
t_res = torch.randint(0, 10, (64, 2, 30, 30)) / 1.0

In [ ]:
print(f"{t_c.shape=}")
print(f"{t_res.shape=}")

In [ ]:
t_c_flat = rearrange(t_c, "b n c h w -> b n (c h w)")
t_res_flat = rearrange(t_res, "b c h w -> b (c h w)")
res = pes(t_c_flat, t_res_flat, mode="complete")
res.shape

In [ ]:
t_c_var = rearrange(t_c, "b n c h w -> b c n (h w)")
t_res_var = rearrange(t_res, "b c h w -> b c (h w)")
res_var = pes(t_c_var, t_res_var, mode="per_var")
res_var.shape

In [ ]:
t_c